In [43]:
import json
import requests
import time
import os

In [ ]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")  # set this in env

GROUND_TRUTH_FILE = "./manual_transcription.json"
PREDICTIONS_FILE = "./qwen3_32b_direct.json"
OUTPUT_FILE = "evaluation_results/qwen3_32b.json"

MODEL_NAME = "openai/gpt-4o-mini"

In [45]:
with open(GROUND_TRUTH_FILE, "r", encoding="utf-8") as f:
    ground_truth_data = json.load(f)

with open(PREDICTIONS_FILE, "r", encoding="utf-8") as f:
    predictions_data = json.load(f)

# Convert to dictionaries for fast lookup
gt_dict = {item["date"]: item["prediction"] for item in ground_truth_data}
pred_dict = {item["date"]: item["prediction"] for item in predictions_data}

evaluation_results = []

In [46]:
SYSTEM_PROMPT = """
You are a deterministic evaluation engine.

You must compute scores strictly using the formulas below.
Do NOT use heuristic grading.
Do NOT round to multiples of 5.
All scores must be integers.
Any integer within the allowed range is valid.
"""

In [47]:
def build_user_prompt(gt_json, pred_json):
    return f"""
EVALUATION METRICS (STRICT)

1. Word Accuracy (0–40 points)

- Compare all dialogue text from the Manual Transcription with the Prediction.
- Ignore panel placement for this metric.
- Ignore case and spacing differences.

Steps:
1. Tokenize dialogue by splitting on whitespace.
2. Let:
      total_words = total number of words in Manual Transcription
      correct_words = number of words from Manual Transcription that appear in Prediction
3. Compute:
      score = (correct_words / total_words) × 40
4. Round to nearest integer.

--------------------------------------------

2. Speaker Accuracy (0–25 points)

For each panel in the Manual Transcription:

- Let:
      manual_speakers = number of speaker entries in that panel
      correct_speakers = number of prediction entries in that same panel
                         where BOTH speaker name and dialogue text match sufficiently
                         (≥ 50% word overlap ignoring case & punctuation)

For that panel:
      panel_ratio = correct_speakers / manual_speakers

After processing all panels:
      average_ratio = mean(panel_ratio across panels)

Final score:
      speaker_score = average_ratio × 25

Round to nearest integer.

--------------------------------------------

3. Panel Structure (0–20 points)

A. Panel Count (0–10 points)

Let:
      manual_panels = number of panels in Manual Transcription
      predicted_panels = number of panels in Prediction

panel_ratio = min(manual_panels, predicted_panels) 
              / max(manual_panels, predicted_panels)

panel_count_score = panel_ratio × 10
Round to nearest integer.

--------------------------------------------

B. Dialogue Placement (0–10 points)

For each dialogue entry in Manual Transcription:

1. Look inside the predicted panel with the same panel number.
2. Compare dialogue texts using word overlap ratio:
      overlap_ratio =
          (# shared words ignoring case & punctuation)
          / (# words in Manual dialogue)

3. If overlap_ratio ≥ 0.7,
      count as correctly placed.

Let:
      correct_placements = number of correctly placed dialogues
      total_dialogues = total dialogues in Manual Transcription

placement_ratio = correct_placements / total_dialogues
placement_score = placement_ratio × 10

Round to nearest integer.

--------------------------------------------

Final Panel Structure Score:
      panel_structure = panel_count_score + placement_score

--------------------------------------------

4. Hallucination (0–15 points)

If no hallucination → 15
If 1 hallucinated line → 14
If 2-3 major hallucinated lines → 12–13
If completely invented content (> 75%) → 0–10

--------------------------------------------

Return STRICT JSON only:

{{
  "word_accuracy": integer,
  "speaker_accuracy": integer,
  "panel_structure": integer,
  "hallucination": integer
}}

--------------------------------------------

Manual Transcription:
{json.dumps(gt_json, indent=2)}

--------------------------------------------

Prediction:
{json.dumps(pred_json, indent=2)}
"""


In [48]:
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1:
        return text[start:end+1]
    return None

In [49]:
for date in gt_dict:
    if date not in pred_dict:
        print(f"Missing prediction for {date}")
        continue

    print(f"Evaluating {date}")

    #print(gt_dict[date], pred_dict[date])

    user_prompt = build_user_prompt(gt_dict[date], pred_dict[date])

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL_NAME,
            "temperature": 0,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ]
        }
    )

    if response.status_code != 200:
        print("Request failed:", response.text)
        continue

    response_json = response.json()
    content = response_json["choices"][0]["message"]["content"]

    cleaned = extract_json(content)
    if not cleaned:
        print("Could not extract JSON for", date)
        continue

    try:
        parsed_scores = json.loads(cleaned)
    except json.JSONDecodeError:
        print("Invalid JSON from model for", date)
        continue

    evaluation_results.append({
        "date": date,
        "scores": parsed_scores
    })

    # Save progressively (checkpoint-safe)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(evaluation_results, f, indent=2)

    time.sleep(1)  # avoid rate limits

print("Evaluation complete. Results saved to", OUTPUT_FILE)

Evaluating 1990-01-01
Evaluating 1990-01-02
Evaluating 1990-01-03
Evaluating 1990-01-04
Evaluating 1990-01-05
Evaluating 1990-01-06
Evaluating 1990-01-07
Evaluating 1990-01-08
Evaluating 1990-01-09
Evaluating 1990-01-10
Evaluating 1990-01-11
Evaluating 1990-01-12
Evaluating 1990-01-13
Evaluating 1990-01-14
Evaluating 1990-01-15
Evaluating 1990-01-16
Evaluating 1990-01-17
Evaluating 1990-01-18
Evaluating 1990-01-19
Evaluating 1990-01-20
Evaluating 1990-01-21
Evaluating 1990-01-22
Evaluating 1990-01-23
Evaluating 1990-01-24
Evaluating 1990-01-25
Evaluating 1990-01-26
Evaluating 1990-01-27
Evaluating 1990-01-28
Evaluating 1990-01-29
Evaluating 1990-01-30
Evaluation complete. Results saved to evaluation_results/qwen3_32b_tes.json
